In [ ]:
from plotnine import *
import pandas as pd

In [ ]:
def process_ebe_data(name):
    df_raw = pd.read_csv("Benchmark_Data_convergence/Mavoglurant_Simwork_ebe_" + name + ".csv")
    ebe_Simwork = (df_raw.rename(columns={"id":"ID"})[["ID","CLint","KbBR","KbMU","KbBO","KbAD","KbRB"]])
    ebe_Simwork["algo"] = "Simwork"
    ebe_Simwork["setting"] = name
    ebe_nlmixr = pd.read_csv("Benchmark_Data_convergence/Mavoglurant_nlmixr_ebe_" + name + ".csv")
    ebe_nlmixr["algo"] = "nlmixr"
    ebe_nlmixr["setting"] = name
    df_raw_2 = pd.read_csv("Benchmark_Data_convergence/Mavoglurant_Simwork_surrogate_ebe_" + name + ".csv")
    ebe_Surrogate = (df_raw_2.rename(columns={"id":"ID"})[["ID","CLint","KbBR","KbMU","KbBO","KbAD","KbRB"]])
    ebe_Surrogate["algo"] = "Simwork with surrogate"
    ebe_Surrogate["setting"] = name
    ebe = pd.concat([ebe_Simwork, ebe_nlmixr, ebe_Surrogate])
    descriptors = ["CLint", "KbBR", "KbMU", "KbBO", "KbAD", "KbRB"]
    ebe = ebe.melt(
        id_vars=["ID", "algo", "setting"], value_vars=descriptors, var_name="descriptor"
    ).reset_index(drop = True)

    return ebe



In [ ]:
ebe_obs_PDU = process_ebe_data("PDU")
ebe_obs_MI = process_ebe_data("MI")
ebe_obs = pd.concat([ebe_obs_PDU, ebe_obs_MI])
display(ebe_obs.head())

In [ ]:
final_fig_size = (12, 10)
color_scale = "Accent"

In [ ]:
p1 = (
    ggplot(ebe_obs, aes(x="value", fill="algo", color="algo"))
    + geom_density(alpha=0.75)
    + facet_grid(
        cols="descriptor",
        rows="setting",
        scales="free",
    )
    + scale_x_continuous(trans="log10")
    + coord_cartesian(ylim=(0, 10))
    + theme_minimal()
    + labs(
        x="Parameter value",
        y="Distribution density",
        color="Source",
        fill="Source",
        subtitle="(B) Individual parameter estimates for Mavoglurant model",
    )
    + scale_fill_cmap_d(color_scale)
    + scale_color_cmap_d(color_scale)
    + theme(
        figure_size=final_fig_size,
        plot_background=element_rect(fill="none"),
        legend_position="bottom",
    )
)
p1